In [2]:
%pip install pandas numpy matplotlib seaborn scipy statsmodels

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\geeta\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [ ]:

# RNA-Seq Differential Gene Expression Analysis
# Dataset: Gene Expression Cancer RNA-Seq (PANCAN)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------

data = pd.read_csv("data.csv")
labels = pd.read_csv("labels.csv")

print("Expression data shape:", data.shape)
print("Labels shape:", labels.shape)
print("\nExpression data:")
print(data.head())
print("\nLabel columns:", labels.columns.tolist())
print(labels.head())

# ------------------------------------------------
# 2. PREPARE GENE EXPRESSION MATRIX
# Samples are rows; genes are columns
# ------------------------------------------------

# Select cancer-type column from labels
possible_label_cols = [
    c for c in labels.columns
    if any(word in c.lower()
           for word in ["class", "cancer", "label", "type"])
]

if not possible_label_cols:
    raise ValueError(
        "Cancer label column not found. Check labels.columns "
        "and set label_col manually."
    )

label_col = possible_label_cols[0]

# Convert expression columns to numeric
expression = data.apply(pd.to_numeric, errors="coerce")

if len(expression) != len(labels):
    raise ValueError(
        "Expression rows and labels do not match. "
        "Check sample IDs before continuing."
    )

# Align rows only if both files are already in the same sample order
expression = expression.reset_index(drop=True)
labels = labels.reset_index(drop=True)

y = labels[label_col].astype(str).str.strip()

print("\nCancer types:")
print(y.value_counts())

# ------------------------------------------------
# 3. SELECT TWO CANCER GROUPS
# BRCA = Breast cancer
# LUAD = Lung adenocarcinoma
# ------------------------------------------------

group1 = "BRCA"
group2 = "LUAD"

selected = y.isin([group1, group2])

X = expression.loc[selected].copy()
y_selected = y.loc[selected].reset_index(drop=True)
X = X.reset_index(drop=True)

# Remove genes containing missing/non-numeric values
X = X.dropna(axis=1)

# Remove genes with no variation
X = X.loc[:, X.nunique() > 1]

print("\nSelected samples:", len(X))
print("Genes analysed:", X.shape[1])
print(y_selected.value_counts())

if (y_selected == group1).sum() < 2 or (
    y_selected == group2
).sum() < 2:
    raise ValueError("At least two samples per group are required.")

# ------------------------------------------------
# 4. DIFFERENTIAL EXPRESSION ANALYSIS
# Welch's t-test on provided normalized values
# ------------------------------------------------

results = []

for gene in X.columns:
    a = X.loc[y_selected == group1, gene].astype(float)
    b = X.loc[y_selected == group2, gene].astype(float)

    test = ttest_ind(a, b, equal_var=False)

    # This log2 ratio is descriptive, not DESeq2 log2FC
    log2fc = np.log2(
        (a.mean() + 1e-8) / (b.mean() + 1e-8)
    ) if a.mean() >= 0 and b.mean() >= 0 else np.nan

    results.append({
        "Gene": gene,
        f"Mean_{group1}": a.mean(),
        f"Mean_{group2}": b.mean(),
        "Log2_Ratio": log2fc,
        "PValue": test.pvalue
    })

deg = pd.DataFrame(results)

# Correct for multiple testing
deg["Adjusted_PValue"] = multipletests(
    deg["PValue"].fillna(1),
    method="fdr_bh"
)[1]

# Significance criteria
deg["Regulation"] = "Not Significant"

sig = (
    (deg["Adjusted_PValue"] < 0.05) &
    (deg["Log2_Ratio"].abs() >= 1)
)

deg.loc[
    sig & (deg["Log2_Ratio"] > 0), "Regulation"
] = f"Higher in {group1}"

deg.loc[
    sig & (deg["Log2_Ratio"] < 0), "Regulation"
] = f"Higher in {group2}"

deg = deg.sort_values("Adjusted_PValue")

print("\nTop 10 genes:")
print(deg.head(10).to_string(index=False))

print("\nSignificant genes:", sig.sum())
print("Higher in", group1, ":", (
    deg["Regulation"] == f"Higher in {group1}"
).sum())
print("Higher in", group2, ":", (
    deg["Regulation"] == f"Higher in {group2}"
).sum())

# Save results
deg.to_csv("differential_expression_results.csv", index=False)

deg[deg["Regulation"] != "Not Significant"].to_csv(
    "significant_genes.csv", index=False
)

# ------------------------------------------------
# 5. VOLCANO PLOT
# ------------------------------------------------

plot_df = deg.copy()
plot_df["MinusLog10FDR"] = -np.log10(
    plot_df["Adjusted_PValue"].clip(lower=1e-300)
)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=plot_df,
    x="Log2_Ratio",
    y="MinusLog10FDR",
    hue="Regulation",
    alpha=0.7
)
plt.axvline(1, linestyle="--")
plt.axvline(-1, linestyle="--")
plt.axhline(-np.log10(0.05), linestyle="--")
plt.title(f"Differential Expression: {group1} vs {group2}")
plt.xlabel("Descriptive log2 mean-expression ratio")
plt.ylabel("-log10(adjusted p-value)")
plt.tight_layout()
plt.show()

# ------------------------------------------------
# 6. HEATMAP OF TOP SIGNIFICANT GENES
# ------------------------------------------------

top = deg[deg["Regulation"] != "Not Significant"].head(20)

if len(top) > 0:
    heat = X[top["Gene"].tolist()].copy()

    # Standardize each gene for visualization
    heat = (heat - heat.mean()) / heat.std().replace(0, 1)
    heat.index = y_selected.values

    plt.figure(figsize=(12, 6))
    sns.heatmap(heat.T, cmap="vlag", center=0)
    plt.title("Top Significant Genes: Expression Heatmap")
    plt.xlabel("Samples (ordered by cancer group)")
    plt.ylabel("Gene features")
    plt.tight_layout()
    plt.show()
else:
    print("No genes met both significance thresholds.")

# ------------------------------------------------
# 7. REPORT SUMMARY
# ------------------------------------------------

print("\nRNA-SEQ ANALYSIS SUMMARY")
print("Comparison:", group1, "vs", group2)
print("Total genes tested:", len(deg))
print("Significant genes:", (
    (deg["Adjusted_PValue"] < 0.05) &
    (deg["Log2_Ratio"].abs() >= 1)
).sum())

print("\nSaved files:")
print("1. differential_expression_results.csv")
print("2. significant_genes.csv")


Expression data shape: (801, 20532)
Labels shape: (801, 2)

Expression data:
  Unnamed: 0  gene_0    gene_1    gene_2    gene_3     gene_4  gene_5  \
0   sample_0     0.0  2.017209  3.265527  5.478487  10.431999     0.0   
1   sample_1     0.0  0.592732  1.588421  7.586157   9.623011     0.0   
2   sample_2     0.0  3.511759  4.327199  6.881787   9.870730     0.0   
3   sample_3     0.0  3.663618  4.507649  6.659068  10.196184     0.0   
4   sample_4     0.0  2.655741  2.821547  6.539454   9.738265     0.0   

     gene_6    gene_7  gene_8  ...  gene_20521  gene_20522  gene_20523  \
0  7.175175  0.591871     0.0  ...    4.926711    8.210257    9.723516   
1  6.816049  0.000000     0.0  ...    4.593372    7.323865    9.740931   
2  6.972130  0.452595     0.0  ...    5.125213    8.127123   10.908640   
3  7.843375  0.434882     0.0  ...    6.076566    8.792959   10.141520   
4  6.566967  0.360982     0.0  ...    5.996032    8.891425   10.373790   

   gene_20524  gene_20525  gene_20526  